# LSTM Baseline Notebook
This notebook implements and trains the baseline encoder model under our unified, leakage-safe LOBench replication pipeline.

In [1]:
# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

Mounted at /content/drive
Mounted Google Drive and changed directory to baselines.


In [2]:
# Install PyTorch Lightning if it is not present in the environment
!pip install -q lightning pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 53.7 MB/s eta 0:00:00


In [3]:
from common import *
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Libraries and common module imported successfully.')

Libraries and common module imported successfully.


In [4]:
class LSTMEncoder(nn.Module):
    def __init__(self, n_features=40, hidden_size=128, num_layers=3, latent_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size,
                             num_layers=num_layers, batch_first=True)
        self.proj = nn.Linear(hidden_size, latent_dim)

    def forward(self, x):  # x: [B, 100, 40]
        out, (hn, cn) = self.lstm(x)
        last = out[:, -1, :]        # [B, hidden_size] -- final timestep
        return self.proj(last)      # [B, latent_dim]

In [5]:
model_name = 'LSTM'
stocks = ['sz000001', 'sz000002', 'sz000858', 'sz300147', 'sz002415']
for stock in stocks:
    print(f'\n========================================')
    print(f'Starting experiment for Model: {model_name} | Stock: {stock}')
    print(f'========================================')
    run_experiment(
        encoder_class=LSTMEncoder,
        model_name=model_name,
        stock=stock,
        latent_dim=256,
        max_epochs=100
    )


Starting experiment for Model: LSTM | Stock: sz000001
Loading data from data/sz000001-level10_processed.csv...
Loaded shape: (1171534, 41)
Columns and Level 10 ordering verified successfully.
Detecting sessions (3-second continuous chunks)...
Detected 488 sessions.
Splitting train/val/test splits (80/10/10 by calendar dates)...
Constructing sequence window indices (seq_len=100, stride=1)...

--- Train/Val/Test Splits & Windows ---
Split      | Rows       | Window Count
Train      | 936254     | 897644
Validation | 115231     | 110479
Test       | 120049     | 115099
---------------------------------------

Encoder parameters: 384,256
Shared Decoder parameters: 4,756,896
Total model parameters: 5,141,152


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Found existing checkpoint at checkpoints/LSTM/sz000001/last.ckpt. Resuming training...


/usr/local/lib/python3.12/dist-packages/lightning/fabric/loggers/csv_logs.py:268: Experiment logs directory training_logs/LSTM/sz000001 exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/LSTM/sz000001 exists and is not empty.
INFO: Restoring states from the checkpoint path at checkpoints/LSTM/sz000001/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at checkpoints/LSTM/sz000001/last.ckpt
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/LSTM/sz000001' to '/content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/LSTM/sz000001', therefore `best_model_s

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ LSTMEncoder   │  384 K │ train │     0 │
│ 1 │ decoder │ SharedDecoder │  4.8 M │ train │     0 │
│ 2 │ loss_fn │ MSELoss       │      0 │ train │     0 │
│ 3 │ mae_fn  │ L1Loss        │      0 │ train │     0 │
└───┴─────────┴───────────────┴────────┴───────┴───────┘

...

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training loop finished in 13943 seconds.
Loading best checkpoint for evaluation: checkpoints/LSTM/sz002415/best.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mae          │    0.11795918643474579    │
│         test_mse          │    0.07764482498168945    │
└───────────────────────────┴───────────────────────────┘


=== FINAL REPORT ===
Model: LSTM | Stock: sz002415
Encoder params: 384,256
Total params (encoder + shared decoder): 5,141,152
Training time: 13943s
Test MSE: 0.0776
Test MAE: 0.1180

